In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import tifffile
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.predict import load_predict_config, load_predictor

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
AXIS_NAMES = ("axis 0", "axis 1", "axis 2")

In [ ]:
config = load_predict_config(PREDICT_CONFIG)
run_dir = ROOT / config.run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = load_predictor(run_dir, device=device)
opts = config.make_options(predictor)
if opts.volume_size == predictor.image_size:
    raise ValueError("Set scale.enabled to true in config/predict.yaml.")

start = time.perf_counter()
volume, stats = predictor.predict(opts)
elapsed = time.perf_counter() - start

out = run_dir / "predictions" / f"mpdd_{opts.volume_size}.tif"
out.parent.mkdir(parents=True, exist_ok=True)
tifffile.imwrite(out, volume.numpy(), photometric="minisblack")

fractions = [round(value, 4) for value in stats["phase_fractions"]]
print(f"shape={tuple(volume.shape)} elapsed={elapsed:.1f}s")
print(f"fractions={fractions} tiff={out}")

In [ ]:
indices = torch.linspace(0, opts.volume_size - 1, 5).round().int().tolist()
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for axis, row in enumerate(axes):
    for index, ax in zip(indices, row):
        ax.imshow(
            volume.select(axis, index),
            cmap="gray",
            vmin=0,
            vmax=opts.num_phases - 1,
            interpolation="nearest",
        )
        ax.set_title(f"{AXIS_NAMES[axis]} · slice {index}", fontsize=12)
        ax.axis("off")
plt.tight_layout()

In [ ]:
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(volume.numpy(), name="MPDD volume")
viewer